# Batch & Loss Calculation Verification Notebook

This notebook verifies that:
1. Tensor batching operations (squeeze/unsqueeze) are correct
2. Legal action masking works properly
3. Loss calculations with linear weighting are correct
4. The `regrets_dict_to_tensor` mapping is accurate

These are critical for correct Deep CFR training.

In [14]:
import sys
import os
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(os.getcwd()))))
sys.path.insert(0, os.path.dirname(os.path.abspath(os.getcwd())))
sys.path.insert(0, os.path.abspath(os.getcwd()).replace('/tests', ''))

import torch
import numpy as np
from typing import Dict, List
print(f"PyTorch version: {torch.__version__}")
print(f"Working directory: {os.getcwd()}")

PyTorch version: 2.9.1
Working directory: /Users/nikhileshbelulkar/Documents/mit-poker-2026/deep_CFR_vNB_integration/tests


In [15]:
# Import our modules
from network.model import DeepCFRModule
from utils.action_mapping import (
    NETWORK_ACTION_TYPES, NUM_ACTIONS, regrets_dict_to_tensor,
    map_network_output_to_actions
)
from utils.infoset_parser import parse_infoset_to_network_input, batch_parse_infosets
from core.trainer import TrainingSample, DeepCFRTrainer
from core.mccfr import MCCFR

print("All imports successful!")
print(f"Number of network actions: {NUM_ACTIONS}")
print(f"Action types: {NETWORK_ACTION_TYPES}")

All imports successful!
Number of network actions: 19
Action types: ['DISCARD_0', 'DISCARD_1', 'DISCARD_2', 'CHECK', 'CALL', 'FOLD', 'RAISE_25_POT', 'RAISE_50_POT', 'RAISE_75_POT', 'RAISE_100_POT', 'RAISE_150_POT', 'RAISE_200_POT', 'RAISE_250_POT', 'RAISE_300_POT', 'RAISE_350_POT', 'RAISE_400_POT', 'RAISE_450_POT', 'RAISE_500_POT', 'RAISE_ALL_IN']


## 1. Test `regrets_dict_to_tensor` Mapping

This function converts action->regret dictionaries to tensors. We verify:
- Correct ordering (matches NETWORK_ACTION_TYPES)
- Missing actions default to 0.0
- Output shape is [19]

In [16]:
def test_regrets_dict_to_tensor():
    """Test that regrets_dict_to_tensor correctly maps action keys to tensor indices."""
    print("=" * 60)
    print("TEST 1: regrets_dict_to_tensor Mapping")
    print("=" * 60)
    
    # Test 1: Full mapping with all actions
    full_regrets = {
        'DISCARD_0': 1.0,
        'DISCARD_1': 2.0,
        'DISCARD_2': 3.0,
        'CHECK': 4.0,
        'CALL': 5.0,
        'FOLD': 6.0,
        'RAISE_25_POT': 7.0,
        'RAISE_50_POT': 8.0,
        'RAISE_75_POT': 9.0,
        'RAISE_100_POT': 10.0,
        'RAISE_150_POT': 11.0,
        'RAISE_200_POT': 12.0,
        'RAISE_250_POT': 13.0,
        'RAISE_300_POT': 14.0,
        'RAISE_350_POT': 15.0,
        'RAISE_400_POT': 16.0,
        'RAISE_450_POT': 17.0,
        'RAISE_500_POT': 18.0,
        'RAISE_ALL_IN': 19.0
    }
    
    tensor = regrets_dict_to_tensor(full_regrets)
    print(f"\n[1a] Full mapping test:")
    print(f"  Input: 19 actions with values 1.0 to 19.0")
    print(f"  Output shape: {tensor.shape}")
    print(f"  Output: {tensor.tolist()}")
    
    # Verify each index
    all_correct = True
    for i, action in enumerate(NETWORK_ACTION_TYPES):
        expected = full_regrets[action]
        actual = tensor[i].item()
        if expected != actual:
            print(f"  ❌ Index {i} ({action}): expected {expected}, got {actual}")
            all_correct = False
    
    if all_correct:
        print(f"  ✅ All 19 indices correctly mapped!")
    
    # Test 2: Partial mapping (only betting actions, typical flop/turn/river)
    betting_regrets = {
        'CHECK': 0.5,
        'FOLD': -1.0,
        'RAISE_50_POT': 2.0,
        'RAISE_100_POT': 3.0,
        'RAISE_ALL_IN': 1.5
    }
    
    tensor = regrets_dict_to_tensor(betting_regrets)
    print(f"\n[1b] Partial mapping test (betting only):")
    print(f"  Input: {betting_regrets}")
    print(f"  Output: {tensor.tolist()}")
    
    # Verify: discards should be 0.0, specified actions should have values
    assert tensor[0].item() == 0.0, "DISCARD_0 should be 0.0"
    assert tensor[1].item() == 0.0, "DISCARD_1 should be 0.0"
    assert tensor[2].item() == 0.0, "DISCARD_2 should be 0.0"
    assert tensor[3].item() == 0.5, "CHECK should be 0.5"
    assert tensor[5].item() == -1.0, "FOLD should be -1.0"
    assert tensor[7].item() == 2.0, "RAISE_50_POT should be 2.0"
    assert tensor[9].item() == 3.0, "RAISE_100_POT should be 3.0"
    assert tensor[18].item() == 1.5, "RAISE_ALL_IN should be 1.5"
    print(f"  ✅ Partial mapping correct! Missing actions default to 0.0")
    
    # Test 3: Discard-only (typical preflop discard phase)
    discard_regrets = {
        'DISCARD_0': 10.0,
        'DISCARD_1': 5.0,
        'DISCARD_2': 2.0
    }
    
    tensor = regrets_dict_to_tensor(discard_regrets)
    print(f"\n[1c] Discard-only mapping test:")
    print(f"  Input: {discard_regrets}")
    print(f"  Output: {tensor.tolist()}")
    
    assert tensor[0].item() == 10.0, "DISCARD_0 should be 10.0"
    assert tensor[1].item() == 5.0, "DISCARD_1 should be 5.0"
    assert tensor[2].item() == 2.0, "DISCARD_2 should be 2.0"
    assert tensor[3:].tolist() == [0.0] * 16, "All other actions should be 0.0"
    print(f"  ✅ Discard-only mapping correct!")
    
    print("\n" + "=" * 60)
    print("✅ TEST 1 PASSED: regrets_dict_to_tensor works correctly")
    print("=" * 60)
    return True

test_regrets_dict_to_tensor()

TEST 1: regrets_dict_to_tensor Mapping

[1a] Full mapping test:
  Input: 19 actions with values 1.0 to 19.0
  Output shape: torch.Size([19])
  Output: [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0]
  ✅ All 19 indices correctly mapped!

[1b] Partial mapping test (betting only):
  Input: {'CHECK': 0.5, 'FOLD': -1.0, 'RAISE_50_POT': 2.0, 'RAISE_100_POT': 3.0, 'RAISE_ALL_IN': 1.5}
  Output: [0.0, 0.0, 0.0, 0.5, 0.0, -1.0, 0.0, 2.0, 0.0, 3.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.5]
  ✅ Partial mapping correct! Missing actions default to 0.0

[1c] Discard-only mapping test:
  Input: {'DISCARD_0': 10.0, 'DISCARD_1': 5.0, 'DISCARD_2': 2.0}
  Output: [10.0, 5.0, 2.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
  ✅ Discard-only mapping correct!

✅ TEST 1 PASSED: regrets_dict_to_tensor works correctly


True

## 2. Test Legal Action Mask Generation

The mask should be 1.0 for legal actions and 0.0 for illegal actions.
This is derived from the keys present in `target_regrets`.

In [17]:
def test_legal_action_mask():
    """Test that legal action masks are generated correctly."""
    print("=" * 60)
    print("TEST 2: Legal Action Mask Generation")
    print("=" * 60)
    
    # Create a minimal trainer for testing _get_legal_action_mask
    network = DeepCFRModule(
        nhandcards=3, nboardcards=6, n_action_history=20, nresponses=19, dim=64
    )
    mccfr = MCCFR()
    trainer = DeepCFRTrainer(network, mccfr, batch_size=8)
    
    # Test 1: Discard phase (only DISCARD actions legal)
    discard_regrets = {
        'DISCARD_0': 1.0,
        'DISCARD_1': 2.0,
        'DISCARD_2': 0.5
    }
    
    mask = trainer._get_legal_action_mask(street=1, target_regrets=discard_regrets)
    print(f"\n[2a] Discard phase mask:")
    print(f"  Legal actions: {list(discard_regrets.keys())}")
    print(f"  Mask: {mask}")
    
    assert mask[0:3] == [1.0, 1.0, 1.0], "DISCARD actions should be legal"
    assert all(m == 0.0 for m in mask[3:]), "Non-discard actions should be illegal"
    print(f"  ✅ Discard mask correct!")
    
    # Test 2: Betting phase NOT facing bet (CHECK, FOLD, RAISES legal)
    betting_not_facing = {
        'CHECK': 0.5,
        'FOLD': -0.5,
        'RAISE_50_POT': 1.0,
        'RAISE_100_POT': 1.5,
        'RAISE_ALL_IN': 0.8
    }
    
    mask = trainer._get_legal_action_mask(street=3, target_regrets=betting_not_facing)
    print(f"\n[2b] Betting (not facing bet) mask:")
    print(f"  Legal actions: {list(betting_not_facing.keys())}")
    print(f"  Mask: {mask}")
    
    assert mask[0:3] == [0.0, 0.0, 0.0], "DISCARD actions should be illegal"
    assert mask[3] == 1.0, "CHECK should be legal"
    assert mask[4] == 0.0, "CALL should be illegal (not facing bet)"
    assert mask[5] == 1.0, "FOLD should be legal"
    assert mask[7] == 1.0, "RAISE_50_POT should be legal"
    assert mask[9] == 1.0, "RAISE_100_POT should be legal"
    assert mask[18] == 1.0, "RAISE_ALL_IN should be legal"
    print(f"  ✅ Betting (not facing) mask correct!")
    
    # Test 3: Betting phase FACING bet (CALL, FOLD, RAISES legal, no CHECK)
    betting_facing = {
        'CALL': 2.0,
        'FOLD': -1.0,
        'RAISE_25_POT': 0.5,
        'RAISE_75_POT': 1.2,
        'RAISE_ALL_IN': 0.3
    }
    
    mask = trainer._get_legal_action_mask(street=3, target_regrets=betting_facing)
    print(f"\n[2c] Betting (facing bet) mask:")
    print(f"  Legal actions: {list(betting_facing.keys())}")
    print(f"  Mask: {mask}")
    
    assert mask[0:3] == [0.0, 0.0, 0.0], "DISCARD actions should be illegal"
    assert mask[3] == 0.0, "CHECK should be illegal (facing bet)"
    assert mask[4] == 1.0, "CALL should be legal"
    assert mask[5] == 1.0, "FOLD should be legal"
    assert mask[6] == 1.0, "RAISE_25_POT should be legal"
    assert mask[8] == 1.0, "RAISE_75_POT should be legal"
    assert mask[18] == 1.0, "RAISE_ALL_IN should be legal"
    print(f"  ✅ Betting (facing) mask correct!")
    
    print("\n" + "=" * 60)
    print("✅ TEST 2 PASSED: Legal action masks generated correctly")
    print("=" * 60)
    return True

test_legal_action_mask()

TEST 2: Legal Action Mask Generation

[2a] Discard phase mask:
  Legal actions: ['DISCARD_0', 'DISCARD_1', 'DISCARD_2']
  Mask: [1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
  ✅ Discard mask correct!

[2b] Betting (not facing bet) mask:
  Legal actions: ['CHECK', 'FOLD', 'RAISE_50_POT', 'RAISE_100_POT', 'RAISE_ALL_IN']
  Mask: [0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0]
  ✅ Betting (not facing) mask correct!

[2c] Betting (facing bet) mask:
  Legal actions: ['CALL', 'FOLD', 'RAISE_25_POT', 'RAISE_75_POT', 'RAISE_ALL_IN']
  Mask: [0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0]
  ✅ Betting (facing) mask correct!

✅ TEST 2 PASSED: Legal action masks generated correctly


True

## 3. Test Batch Preparation (prepare_batch)

Verify that `prepare_batch` correctly:
- Parses infosets into canonical cards and action histories
- Stacks target tensors with correct shape [batch_size, 19]
- Creates correct iteration weights tensor [batch_size]
- Creates correct legal mask tensor [batch_size, 19]

In [18]:
def test_prepare_batch():
    """Test that prepare_batch creates correctly shaped tensors."""
    print("=" * 60)
    print("TEST 3: Batch Preparation (prepare_batch)")
    print("=" * 60)
    
    # Create network and trainer
    network = DeepCFRModule(
        nhandcards=3, nboardcards=6, n_action_history=20, nresponses=19, dim=64
    )
    mccfr = MCCFR()
    trainer = DeepCFRTrainer(network, mccfr, batch_size=8)
    
    # Create sample training samples with different action sets
    samples = [
        TrainingSample(
            infoset="S1|H:14s0,13s0,10s1|B:|A:",
            target_regrets={'DISCARD_0': 1.0, 'DISCARD_1': 0.5, 'DISCARD_2': 0.2},
            player=0,
            iteration=1
        ),
        TrainingSample(
            infoset="S0|H:14s0,12s1,8s0|B:|A:R",
            target_regrets={'CALL': 2.0, 'FOLD': -1.0, 'RAISE_50_POT': 1.0, 'RAISE_ALL_IN': 0.5},
            player=0,
            iteration=5
        ),
        TrainingSample(
            infoset="S3|H:14s0,13s0,10s1|B:9s0,7s1|A:XRC",
            target_regrets={'CHECK': 0.8, 'FOLD': -0.3, 'RAISE_100_POT': 1.5},
            player=1,
            iteration=10
        )
    ]
    
    batch_size = len(samples)
    
    # Call prepare_batch
    cc_list, ah_list, target_batch, iter_weights, legal_mask = trainer.prepare_batch(samples)
    
    print(f"\n[3a] Batch dimensions:")
    print(f"  Batch size: {batch_size}")
    print(f"  Canon cards list length: {len(cc_list)}")
    print(f"  Action history list length: {len(ah_list)}")
    print(f"  Target batch shape: {target_batch.shape}")
    print(f"  Iteration weights shape: {iter_weights.shape}")
    print(f"  Legal mask shape: {legal_mask.shape}")
    
    assert len(cc_list) == batch_size, "Canon cards list should match batch size"
    assert len(ah_list) == batch_size, "Action history list should match batch size"
    assert target_batch.shape == (batch_size, 19), f"Target should be [batch, 19], got {target_batch.shape}"
    assert iter_weights.shape == (batch_size,), f"Weights should be [batch], got {iter_weights.shape}"
    assert legal_mask.shape == (batch_size, 19), f"Mask should be [batch, 19], got {legal_mask.shape}"
    print(f"  ✅ All dimensions correct!")
    
    print(f"\n[3b] Target tensor values:")
    for i, sample in enumerate(samples):
        print(f"  Sample {i}: {sample.target_regrets}")
        print(f"    Target tensor: {target_batch[i].tolist()[:6]}... (first 6)")
    
    # Verify target tensor values match regrets (use abs() for float32 precision)
    # Sample 0: discard regrets
    assert target_batch[0, 0].item() == 1.0, "DISCARD_0 regret should be 1.0"
    assert abs(target_batch[0, 1].item() - 0.5) < 1e-4, "DISCARD_1 regret should be ~0.5"
    assert abs(target_batch[0, 2].item() - 0.2) < 1e-4, "DISCARD_2 regret should be ~0.2"
    print(f"  ✅ Sample 0 target values correct!")
    
    # Sample 1: betting regrets
    assert target_batch[1, 4].item() == 2.0, "CALL regret should be 2.0"
    assert target_batch[1, 5].item() == -1.0, "FOLD regret should be -1.0"
    assert target_batch[1, 7].item() == 1.0, "RAISE_50_POT regret should be 1.0"
    assert abs(target_batch[1, 18].item() - 0.5) < 1e-4, "RAISE_ALL_IN regret should be ~0.5"
    print(f"  ✅ Sample 1 target values correct!")
    
    print(f"\n[3c] Iteration weights:")
    print(f"  Weights: {iter_weights.tolist()}")
    assert iter_weights[0].item() == 1, "Iteration 1"
    assert iter_weights[1].item() == 5, "Iteration 5"
    assert iter_weights[2].item() == 10, "Iteration 10"
    print(f"  ✅ Iteration weights correct!")
    
    print(f"\n[3d] Legal action masks:")
    for i, sample in enumerate(samples):
        legal_count = legal_mask[i].sum().item()
        print(f"  Sample {i}: {int(legal_count)} legal actions")
        print(f"    Mask: {legal_mask[i].tolist()[:6]}... (first 6)")
    
    # Verify masks match legal actions
    # Sample 0: only discards legal
    assert legal_mask[0, 0:3].sum().item() == 3, "All 3 discards legal"
    assert legal_mask[0, 3:].sum().item() == 0, "No betting actions legal"
    print(f"  ✅ Sample 0 mask correct!")
    
    # Sample 1: CALL, FOLD, RAISE_50_POT, RAISE_ALL_IN legal
    assert legal_mask[1, 4].item() == 1.0, "CALL legal"
    assert legal_mask[1, 5].item() == 1.0, "FOLD legal"
    assert legal_mask[1, 7].item() == 1.0, "RAISE_50_POT legal"
    assert legal_mask[1, 18].item() == 1.0, "RAISE_ALL_IN legal"
    assert legal_mask[1, 3].item() == 0.0, "CHECK illegal (facing bet)"
    print(f"  ✅ Sample 1 mask correct!")
    
    print("\n" + "=" * 60)
    print("✅ TEST 3 PASSED: prepare_batch works correctly")
    print("=" * 60)
    return True

test_prepare_batch()

TEST 3: Batch Preparation (prepare_batch)

[3a] Batch dimensions:
  Batch size: 3
  Canon cards list length: 3
  Action history list length: 3
  Target batch shape: torch.Size([3, 19])
  Iteration weights shape: torch.Size([3])
  Legal mask shape: torch.Size([3, 19])
  ✅ All dimensions correct!

[3b] Target tensor values:
  Sample 0: {'DISCARD_0': 1.0, 'DISCARD_1': 0.5, 'DISCARD_2': 0.2}
    Target tensor: [1.0, 0.5, 0.20000000298023224, 0.0, 0.0, 0.0]... (first 6)
  Sample 1: {'CALL': 2.0, 'FOLD': -1.0, 'RAISE_50_POT': 1.0, 'RAISE_ALL_IN': 0.5}
    Target tensor: [0.0, 0.0, 0.0, 0.0, 2.0, -1.0]... (first 6)
  Sample 2: {'CHECK': 0.8, 'FOLD': -0.3, 'RAISE_100_POT': 1.5}
    Target tensor: [0.0, 0.0, 0.0, 0.800000011920929, 0.0, -0.30000001192092896]... (first 6)
  ✅ Sample 0 target values correct!
  ✅ Sample 1 target values correct!

[3c] Iteration weights:
  Weights: [1.0, 5.0, 10.0]
  ✅ Iteration weights correct!

[3d] Legal action masks:
  Sample 0: 3 legal actions
    Mask: [1.0, 1

True

## 4. Test Network Forward Pass Dimensions

Verify that the network handles batches correctly:
- Single sample: output shape [1, 19]
- Multiple samples: output shape [batch_size, 19]
- Squeeze/unsqueeze operations preserve batch dimension

In [19]:
def test_network_batch_dimensions():
    """Test that network forward pass handles batch dimensions correctly."""
    print("=" * 60)
    print("TEST 4: Network Batch Dimensions")
    print("=" * 60)
    
    # Create network
    network = DeepCFRModule(
        nhandcards=3, nboardcards=6, n_action_history=20, nresponses=19, dim=64
    )
    network.eval()
    
    # Parse some test infosets
    infosets = [
        "S0|H:14s0,13s0,10s1|B:|A:",
        "S1|H:12s0,11s1,8s0|B:|A:C",
        "S3|H:14s0,13s0,10s1|B:9s0,7s1|A:XRC",
        "S3|H:10s0,9s0,8s0|B:7s1,6s1,5s0|A:RRCX"
    ]
    
    cc_list, ah_list = batch_parse_infosets(infosets)
    
    # Test batch forward pass
    print(f"\n[4a] Batch forward pass (batch_size={len(infosets)}):")
    with torch.no_grad():
        output = network(cc_list, ah_list)
    
    print(f"  Input: {len(cc_list)} canonical card objects, {len(ah_list)} action histories")
    print(f"  Output shape: {output.shape}")
    print(f"  Output dtype: {output.dtype}")
    
    assert output.shape == (len(infosets), 19), f"Expected [{len(infosets)}, 19], got {output.shape}"
    print(f"  ✅ Batch output shape correct!")
    
    # Test single sample (should also work)
    print(f"\n[4b] Single sample forward pass:")
    cc_single, ah_single = batch_parse_infosets([infosets[0]])
    with torch.no_grad():
        output_single = network(cc_single, ah_single)
    
    print(f"  Input: 1 sample")
    print(f"  Output shape: {output_single.shape}")
    
    assert output_single.shape == (1, 19), f"Expected [1, 19], got {output_single.shape}"
    print(f"  ✅ Single sample output shape correct!")
    
    # Verify outputs are different for different inputs (network learned something)
    print(f"\n[4c] Output variation check:")
    for i in range(len(infosets)):
        print(f"  Sample {i} output mean: {output[i].mean().item():.4f}, std: {output[i].std().item():.4f}")
    
    # Check that outputs aren't all identical
    output_means = [output[i].mean().item() for i in range(len(infosets))]
    if len(set([round(m, 4) for m in output_means])) == 1:
        print(f"  ⚠️ Warning: All outputs have same mean (network may not be differentiating)")
    else:
        print(f"  ✅ Network produces different outputs for different inputs!")
    
    print("\n" + "=" * 60)
    print("✅ TEST 4 PASSED: Network batch dimensions correct")
    print("=" * 60)
    return True

test_network_batch_dimensions()

TEST 4: Network Batch Dimensions

[4a] Batch forward pass (batch_size=4):
  Input: 4 canonical card objects, 4 action histories
  Output shape: torch.Size([4, 19])
  Output dtype: torch.float32
  ✅ Batch output shape correct!

[4b] Single sample forward pass:
  Input: 1 sample
  Output shape: torch.Size([1, 19])
  ✅ Single sample output shape correct!

[4c] Output variation check:
  Sample 0 output mean: -0.0002, std: 0.0128
  Sample 1 output mean: 0.0006, std: 0.0125
  Sample 2 output mean: -0.0009, std: 0.0122
  Sample 3 output mean: 0.0024, std: 0.0116
  ✅ Network produces different outputs for different inputs!

✅ TEST 4 PASSED: Network batch dimensions correct


True

## 5. Test Loss Calculation with Legal Action Masking

This is the CRITICAL test. We verify:
1. Squared errors are computed correctly
2. Legal mask zeros out illegal actions
3. Per-sample loss is sum over legal actions only
4. Linear weighting scales losses correctly
5. LCFR 2/T rescaling works

In [20]:
def test_loss_calculation():
    """Test that loss calculation with masking and weighting is correct."""
    print("=" * 60)
    print("TEST 5: Loss Calculation with Masking & Weighting")
    print("=" * 60)
    
    torch.manual_seed(42)  # For reproducibility
    
    # Create simple test tensors
    batch_size = 3
    num_actions = 19
    
    # Predictions (network output)
    predictions = torch.tensor([
        [1.0, 2.0, 3.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],  # Discard phase
        [0.0, 0.0, 0.0, 0.0, 2.5, 1.0, 0.0, 1.5, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.5],  # Facing bet
        [0.0, 0.0, 0.0, 1.0, 0.0, 0.5, 0.0, 0.0, 0.0, 2.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],  # Not facing bet
    ], dtype=torch.float32)
    
    # Targets (ground truth regrets)
    targets = torch.tensor([
        [2.0, 1.0, 4.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],  # Different from pred
        [0.0, 0.0, 0.0, 0.0, 3.0, 0.5, 0.0, 2.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0],  # Different
        [0.0, 0.0, 0.0, 1.5, 0.0, 0.0, 0.0, 0.0, 0.0, 2.5, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],  # Different
    ], dtype=torch.float32)
    
    # Legal masks
    legal_mask = torch.tensor([
        [1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],  # Only discards
        [0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0],  # CALL, FOLD, R50, ALL_IN
        [0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],  # CHECK, FOLD, R100
    ], dtype=torch.float32)
    
    # Iteration weights
    iter_weights = torch.tensor([1.0, 5.0, 10.0], dtype=torch.float32)
    
    print(f"\n[5a] Manual loss calculation:")
    print(f"  Predictions shape: {predictions.shape}")
    print(f"  Targets shape: {targets.shape}")
    print(f"  Legal mask shape: {legal_mask.shape}")
    
    # Step 1: Compute squared errors
    squared_errors = (predictions - targets) ** 2
    print(f"\n  Step 1: Squared errors")
    print(f"    Sample 0 (discards): {squared_errors[0, :3].tolist()}")
    print(f"    Sample 1 (betting):  {squared_errors[1, [4,5,7,18]].tolist()}")
    print(f"    Sample 2 (not facing): {squared_errors[2, [3,5,9]].tolist()}")
    
    # Step 2: Apply mask
    masked_errors = squared_errors * legal_mask
    print(f"\n  Step 2: Masked errors (illegal actions zeroed)")
    print(f"    Sample 0 non-zero: {masked_errors[0].nonzero().squeeze().tolist()}")
    print(f"    Sample 1 non-zero: {masked_errors[1].nonzero().squeeze().tolist()}")
    print(f"    Sample 2 non-zero: {masked_errors[2].nonzero().squeeze().tolist()}")
    
    # Step 3: Per-sample loss (sum over actions)
    per_sample_loss = masked_errors.sum(dim=1)
    print(f"\n  Step 3: Per-sample loss (sum of masked errors)")
    print(f"    Sample 0: {per_sample_loss[0].item():.4f}")
    print(f"    Sample 1: {per_sample_loss[1].item():.4f}")
    print(f"    Sample 2: {per_sample_loss[2].item():.4f}")
    
    # Verify manually for sample 0:
    # pred=[1,2,3], target=[2,1,4] -> errors = [1,1,1] -> sum = 3.0
    expected_sample0 = (1-2)**2 + (2-1)**2 + (3-4)**2  # = 1 + 1 + 1 = 3
    assert abs(per_sample_loss[0].item() - expected_sample0) < 1e-5, f"Sample 0 loss mismatch"
    print(f"    ✅ Sample 0 verified: {expected_sample0}")
    
    # Step 4: Apply iteration weights (linear weighting)
    weighted_loss_per_sample = iter_weights * per_sample_loss
    print(f"\n  Step 4: Weighted loss (iter_weight * per_sample_loss)")
    print(f"    Sample 0 (weight=1): {weighted_loss_per_sample[0].item():.4f}")
    print(f"    Sample 1 (weight=5): {weighted_loss_per_sample[1].item():.4f}")
    print(f"    Sample 2 (weight=10): {weighted_loss_per_sample[2].item():.4f}")
    
    # Step 5: Final batch loss (mean of weighted losses)
    final_loss = weighted_loss_per_sample.mean()
    print(f"\n  Step 5: Final batch loss (mean): {final_loss.item():.4f}")
    
    # Step 6: Test LCFR 2/T rescaling
    current_iteration = 10  # T = 10
    scaled_weights = iter_weights * (2.0 / current_iteration)
    lcfr_loss = (scaled_weights * per_sample_loss).mean()
    print(f"\n  Step 6: LCFR loss (2/T rescaling, T={current_iteration}): {lcfr_loss.item():.4f}")
    print(f"    Scaled weights: {scaled_weights.tolist()}")
    
    print(f"\n[5b] Verify illegal actions don't contribute to loss:")
    # Add error to an illegal action position and verify loss unchanged
    predictions_modified = predictions.clone()
    predictions_modified[0, 10] = 999.0  # Modify illegal action (RAISE_150_POT)
    
    squared_errors_modified = (predictions_modified - targets) ** 2
    masked_errors_modified = squared_errors_modified * legal_mask
    per_sample_loss_modified = masked_errors_modified.sum(dim=1)
    
    assert abs(per_sample_loss_modified[0].item() - per_sample_loss[0].item()) < 1e-5, \
        "Modifying illegal action should not change loss!"
    print(f"  Original loss: {per_sample_loss[0].item():.4f}")
    print(f"  Modified loss: {per_sample_loss_modified[0].item():.4f}")
    print(f"  ✅ Illegal action modification does NOT affect loss!")
    
    print("\n" + "=" * 60)
    print("✅ TEST 5 PASSED: Loss calculation is correct")
    print("=" * 60)
    return True

test_loss_calculation()

TEST 5: Loss Calculation with Masking & Weighting

[5a] Manual loss calculation:
  Predictions shape: torch.Size([3, 19])
  Targets shape: torch.Size([3, 19])
  Legal mask shape: torch.Size([3, 19])

  Step 1: Squared errors
    Sample 0 (discards): [1.0, 1.0, 1.0]
    Sample 1 (betting):  [0.25, 0.25, 0.25, 0.25]
    Sample 2 (not facing): [0.25, 0.25, 0.25]

  Step 2: Masked errors (illegal actions zeroed)
    Sample 0 non-zero: [0, 1, 2]
    Sample 1 non-zero: [4, 5, 7, 18]
    Sample 2 non-zero: [3, 5, 9]

  Step 3: Per-sample loss (sum of masked errors)
    Sample 0: 3.0000
    Sample 1: 1.0000
    Sample 2: 0.7500
    ✅ Sample 0 verified: 3

  Step 4: Weighted loss (iter_weight * per_sample_loss)
    Sample 0 (weight=1): 3.0000
    Sample 1 (weight=5): 5.0000
    Sample 2 (weight=10): 7.5000

  Step 5: Final batch loss (mean): 5.1667

  Step 6: LCFR loss (2/T rescaling, T=10): 1.0333
    Scaled weights: [0.20000000298023224, 1.0, 2.0]

[5b] Verify illegal actions don't contribute

True

## 6. End-to-End Training Step Test

Run a full training step with real samples and verify:
1. Gradients flow correctly
2. Loss decreases with training
3. Predictions change after gradient update

In [21]:
def test_end_to_end_training_step():
    """Test a complete training step from samples to gradient update."""
    print("=" * 60)
    print("TEST 6: End-to-End Training Step")
    print("=" * 60)
    
    torch.manual_seed(42)
    
    # Create network, MCCFR, and trainer
    network = DeepCFRModule(
        nhandcards=3, nboardcards=6, n_action_history=20, nresponses=19, dim=64
    )
    mccfr = MCCFR()
    trainer = DeepCFRTrainer(
        network, mccfr, 
        batch_size=4,
        learning_rate=0.01,  # Higher LR for visible change
        sgd_iterations=10
    )
    
    # Create diverse training samples
    samples = [
        TrainingSample(
            infoset="S1|H:14s0,13s0,10s1|B:|A:",
            target_regrets={'DISCARD_0': 5.0, 'DISCARD_1': -2.0, 'DISCARD_2': 1.0},
            player=0, iteration=1
        ),
        TrainingSample(
            infoset="S0|H:14s0,12s1,8s0|B:|A:R",
            target_regrets={'CALL': 3.0, 'FOLD': -1.0, 'RAISE_100_POT': 2.0},
            player=0, iteration=3
        ),
        TrainingSample(
            infoset="S3|H:14s0,13s0,10s1|B:9s0,7s1|A:XRC",
            target_regrets={'CHECK': 4.0, 'FOLD': -3.0, 'RAISE_50_POT': 1.0, 'RAISE_ALL_IN': 0.5},
            player=1, iteration=5
        ),
        TrainingSample(
            infoset="S2|H:10s0,9s0,8s0|B:7s1|A:D",
            target_regrets={'DISCARD_0': 1.0, 'DISCARD_1': 3.0, 'DISCARD_2': -1.0},
            player=1, iteration=7
        )
    ]
    
    for sample in samples:
        trainer.add_sample(sample)
    
    print(f"\n[6a] Before training:")
    print(f"  Samples in memory: {len(trainer.samples)}")
    
    # Get initial predictions
    cc_list, ah_list, target_batch, iter_weights, legal_mask = trainer.prepare_batch(samples)
    network.eval()
    with torch.no_grad():
        initial_predictions = network(cc_list, ah_list).clone()
    
    print(f"  Initial prediction means: {[f'{p.mean().item():.3f}' for p in initial_predictions]}")
    
    # Compute initial loss
    initial_squared_errors = (initial_predictions - target_batch) ** 2
    initial_masked = initial_squared_errors * legal_mask
    initial_loss = initial_masked.sum(dim=1).mean().item()
    print(f"  Initial loss: {initial_loss:.4f}")
    
    # Run training
    print(f"\n[6b] Training...")
    network.train()
    metrics = trainer.train_on_samples(
        use_fixed_iterations=True,
        use_linear_weighting=True,
        verbose=False,
        current_iteration=10
    )
    
    print(f"  Training complete!")
    print(f"  Avg loss: {metrics['loss']:.4f}")
    print(f"  Loss start: {metrics.get('loss_start', 'N/A')}")
    print(f"  Loss end: {metrics.get('loss_end', 'N/A')}")
    print(f"  Num batches: {metrics['num_batches']}")
    print(f"  Avg grad norm: {metrics['avg_grad_norm']:.4f}")
    
    # Get final predictions
    print(f"\n[6c] After training:")
    network.eval()
    with torch.no_grad():
        final_predictions = network(cc_list, ah_list).clone()
    
    print(f"  Final prediction means: {[f'{p.mean().item():.3f}' for p in final_predictions]}")
    
    # Compute final loss
    final_squared_errors = (final_predictions - target_batch) ** 2
    final_masked = final_squared_errors * legal_mask
    final_loss = final_masked.sum(dim=1).mean().item()
    print(f"  Final loss: {final_loss:.4f}")
    
    # Check predictions changed
    prediction_diff = (final_predictions - initial_predictions).abs().mean().item()
    print(f"\n[6d] Verification:")
    print(f"  Prediction change (mean abs diff): {prediction_diff:.4f}")
    
    if prediction_diff > 1e-4:
        print(f"  ✅ Predictions changed after training!")
    else:
        print(f"  ⚠️ Predictions barely changed - may indicate issue")
    
    if final_loss < initial_loss:
        print(f"  ✅ Loss decreased: {initial_loss:.4f} → {final_loss:.4f}")
    else:
        print(f"  ⚠️ Loss did not decrease (might be OK with small LR)")
    
    print("\n" + "=" * 60)
    print("✅ TEST 6 PASSED: End-to-end training works")
    print("=" * 60)
    return True

test_end_to_end_training_step()

TEST 6: End-to-End Training Step

[6a] Before training:
  Samples in memory: 4
  Initial prediction means: ['0.005', '0.005', '0.005', '0.004']
  Initial loss: 20.3571

[6b] Training...
  Training complete!
  Avg loss: 5.4212
  Loss start: 14.010795593261719
  Loss end: 1.818776249885559
  Num batches: 10
  Avg grad norm: 13.1189

[6c] After training:
  Final prediction means: ['0.566', '0.545', '0.644', '0.600']
  Final loss: 6.4299

[6d] Verification:
  Prediction change (mean abs diff): 0.8537
  ✅ Predictions changed after training!
  ✅ Loss decreased: 20.3571 → 6.4299

✅ TEST 6 PASSED: End-to-end training works


True

## 7. Test Gradient Flow Through Masked Loss

Verify that gradients:
1. Flow correctly through legal actions
2. Are ZERO for illegal actions (due to mask)
3. Scale correctly with iteration weights

In [22]:
def test_gradient_flow():
    """Test that gradients flow correctly through the masked loss."""
    print("=" * 60)
    print("TEST 7: Gradient Flow Through Masked Loss")
    print("=" * 60)
    
    torch.manual_seed(42)
    
    # Create simple scenario for gradient analysis
    network = DeepCFRModule(
        nhandcards=3, nboardcards=6, n_action_history=20, nresponses=19, dim=64
    )
    network.train()
    
    # Simple single-sample batch for clear gradient analysis
    infoset = "S1|H:14s0,13s0,10s1|B:|A:"
    cc_list, ah_list = batch_parse_infosets([infoset])
    
    # Target: only discards have regrets
    target_regrets = {'DISCARD_0': 5.0, 'DISCARD_1': -2.0, 'DISCARD_2': 3.0}
    target_tensor = regrets_dict_to_tensor(target_regrets).unsqueeze(0)  # [1, 19]
    
    # Legal mask: only discards are legal
    legal_mask = torch.zeros(1, 19)
    legal_mask[0, 0:3] = 1.0  # Only DISCARD_0, 1, 2 are legal
    
    # Forward pass
    predictions = network(cc_list, ah_list)  # [1, 19]
    
    print(f"\n[7a] Predictions:")
    print(f"  Shape: {predictions.shape}")
    print(f"  Discard outputs: {predictions[0, :3].tolist()}")
    print(f"  Other outputs: {predictions[0, 3:6].tolist()} (should not affect loss)")
    
    # Compute masked loss (matches trainer.train_on_samples)
    squared_errors = (predictions - target_tensor) ** 2
    masked_errors = squared_errors * legal_mask
    per_sample_loss = masked_errors.sum(dim=1)  # [1]
    loss = per_sample_loss.mean()
    
    print(f"\n[7b] Loss computation:")
    print(f"  Squared errors (discards): {squared_errors[0, :3].tolist()}")
    print(f"  Squared errors (others): {squared_errors[0, 3:6].tolist()}")
    print(f"  Masked errors (discards): {masked_errors[0, :3].tolist()}")
    print(f"  Masked errors (others): {masked_errors[0, 3:6].tolist()} (should be 0!)")
    print(f"  Loss: {loss.item():.4f}")
    
    # Backward pass
    loss.backward()
    
    print(f"\n[7c] Gradient analysis:")
    # Check gradients on the output layer
    output_weight_grad = network.action_head.weight.grad
    output_bias_grad = network.action_head.bias.grad
    
    print(f"  Output bias gradients:")
    print(f"    Discard actions [0:3]: {output_bias_grad[:3].tolist()}")
    print(f"    Other actions [3:6]: {output_bias_grad[3:6].tolist()}")
    
    # The key test: illegal actions should have ZERO gradient in bias
    illegal_grads = output_bias_grad[3:].tolist()
    legal_grads = output_bias_grad[:3].tolist()
    
    # Check that illegal action gradients are zero
    all_illegal_zero = all(abs(g) < 1e-8 for g in illegal_grads)
    any_legal_nonzero = any(abs(g) > 1e-8 for g in legal_grads)
    
    if all_illegal_zero:
        print(f"  ✅ All illegal action gradients are ZERO!")
    else:
        print(f"  ❌ Some illegal action gradients are non-zero!")
        print(f"     Values: {illegal_grads[:5]}...")
    
    if any_legal_nonzero:
        print(f"  ✅ Legal action gradients are NON-ZERO!")
    else:
        print(f"  ❌ Legal action gradients are all zero (no learning)!")
    
    # Gradient directions should be correct
    # For MSE loss: gradient = 2 * (pred - target) / n
    # If pred < target, gradient should be negative (push pred up)
    print(f"\n[7d] Gradient direction check:")
    for i, name in enumerate(['DISCARD_0', 'DISCARD_1', 'DISCARD_2']):
        pred = predictions[0, i].item()
        target = target_tensor[0, i].item()
        grad = output_bias_grad[i].item()
        expected_direction = "positive" if pred > target else "negative"
        actual_direction = "positive" if grad > 0 else "negative"
        match = "✅" if expected_direction == actual_direction else "❌"
        print(f"  {name}: pred={pred:.2f}, target={target:.2f}, grad={grad:.4f} {match}")
    
    print("\n" + "=" * 60)
    print("✅ TEST 7 PASSED: Gradient flow is correct")
    print("=" * 60)
    return True

test_gradient_flow()

TEST 7: Gradient Flow Through Masked Loss

[7a] Predictions:
  Shape: torch.Size([1, 19])
  Discard outputs: [-0.0064146798104047775, 0.014035794883966446, 0.010074222460389137]
  Other outputs: [0.0022232667542994022, -0.03077400103211403, -0.007839559577405453] (should not affect loss)

[7b] Loss computation:
  Squared errors (discards): [25.064189910888672, 4.056339740753174, 8.939656257629395]
  Squared errors (others): [4.942914983985247e-06, 0.0009470391669310629, 6.145869701867923e-05]
  Masked errors (discards): [25.064189910888672, 4.056339740753174, 8.939656257629395]
  Masked errors (others): [0.0, 0.0, 0.0] (should be 0!)
  Loss: 38.0602

[7c] Gradient analysis:
  Output bias gradients:
    Discard actions [0:3]: [-10.012829780578613, 4.028071403503418, -5.979851722717285]
    Other actions [3:6]: [0.0, 0.0, 0.0]
  ✅ All illegal action gradients are ZERO!
  ✅ Legal action gradients are NON-ZERO!

[7d] Gradient direction check:
  DISCARD_0: pred=-0.01, target=5.00, grad=-10.

True

## 8. Summary and Recommendations

In [23]:
def run_all_tests():
    """Run all tests and print summary."""
    print("\n" + "#" * 70)
    print("# RUNNING ALL BATCH & LOSS CALCULATION TESTS")
    print("#" * 70 + "\n")
    
    tests = [
        ("regrets_dict_to_tensor Mapping", test_regrets_dict_to_tensor),
        ("Legal Action Mask Generation", test_legal_action_mask),
        ("Batch Preparation (prepare_batch)", test_prepare_batch),
        ("Network Batch Dimensions", test_network_batch_dimensions),
        ("Loss Calculation with Masking", test_loss_calculation),
        ("End-to-End Training Step", test_end_to_end_training_step),
        ("Gradient Flow Through Masked Loss", test_gradient_flow),
    ]
    
    results = []
    for name, test_fn in tests:
        try:
            success = test_fn()
            results.append((name, success))
        except Exception as e:
            print(f"\n❌ TEST FAILED: {name}")
            print(f"   Error: {e}")
            import traceback
            traceback.print_exc()
            results.append((name, False))
        print("\n")
    
    # Print summary
    print("\n" + "#" * 70)
    print("# TEST SUMMARY")
    print("#" * 70)
    
    passed = sum(1 for _, success in results if success)
    total = len(results)
    
    for name, success in results:
        status = "✅ PASS" if success else "❌ FAIL"
        print(f"  {status}: {name}")
    
    print(f"\n  Total: {passed}/{total} tests passed")
    
    if passed == total:
        print("\n" + "="*70)
        print("🎉 ALL TESTS PASSED!")
        print("="*70)
        print("\nThe batch operations and loss calculations are working correctly:")
        print("  ✓ regrets_dict_to_tensor maps actions to correct tensor indices")
        print("  ✓ Legal action masks are generated from target_regrets keys")
        print("  ✓ prepare_batch creates correctly shaped tensors")
        print("  ✓ Network handles batch dimensions correctly")
        print("  ✓ Loss is computed only over legal actions (mask works)")
        print("  ✓ Linear weighting and LCFR 2/T rescaling are correct")
        print("  ✓ Gradients flow through legal actions, not illegal ones")
    else:
        print("\n⚠️ Some tests failed - review the output above")
    
    return passed == total

all_passed = run_all_tests()


######################################################################
# RUNNING ALL BATCH & LOSS CALCULATION TESTS
######################################################################

TEST 1: regrets_dict_to_tensor Mapping

[1a] Full mapping test:
  Input: 19 actions with values 1.0 to 19.0
  Output shape: torch.Size([19])
  Output: [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0]
  ✅ All 19 indices correctly mapped!

[1b] Partial mapping test (betting only):
  Input: {'CHECK': 0.5, 'FOLD': -1.0, 'RAISE_50_POT': 2.0, 'RAISE_100_POT': 3.0, 'RAISE_ALL_IN': 1.5}
  Output: [0.0, 0.0, 0.0, 0.5, 0.0, -1.0, 0.0, 2.0, 0.0, 3.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.5]
  ✅ Partial mapping correct! Missing actions default to 0.0

[1c] Discard-only mapping test:
  Input: {'DISCARD_0': 10.0, 'DISCARD_1': 5.0, 'DISCARD_2': 2.0}
  Output: [10.0, 5.0, 2.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
